# Gateway OAuth InboundでAWS LambdaをMCP化する
## Bedrock AgentCore GatewayでAWS Lambda関数を安全なMCPツールに変換する

## 概要
Bedrock AgentCore Gatewayは、既存のAWS Lambda関数を完全管理型のMCPサーバーに変換する方法を提供し、インフラやホスティングの管理を不要にします。Gatewayは、これらすべてのツールにわたって統一されたModel Context Protocol（MCP）インターフェースを提供します。Gatewayは、着信リクエストとターゲットリソースへのアウトバウンド接続の両方に対して安全なアクセス制御を確保するために、デュアル認証モデルを採用しています。フレームワークは2つの主要コンポーネントで構成されています：Gatewayターゲットへのアクセスを試みるユーザーを検証および承認するInbound Authと、認証されたユーザーに代わってGatewayがバックエンドリソースに安全に接続できるようにするOutbound Authです。Gatewayは、アウトバウンド認証のためにAWS Lambda関数への呼び出しを承認するためにIAMロールを使用します。

この例では、インバウンド認証にOAuth、アウトバウンド認証にIAMロールを使用する方法を実演します。

![動作の仕組み](images/lambda-iam-gateway.png)

### チュートリアルの詳細


| 情報              | 詳細                                                       |
|:------------------|:-----------------------------------------------------------|
| チュートリアルタイプ | インタラクティブ                                           |
| AgentCoreコンポーネント | AgentCore Gateway、AgentCore Identity                     |
| エージェントフレームワーク | Strands Agents                                            |
| Gatewayターゲットタイプ | AWS Lambda                                                |
| Inbound Auth IdP  | Amazon Cognito                                            |
| Outbound Auth     | AWS IAM                                                   |
| LLMモデル         | Anthropic Claude Haiku 4.5、Amazon Nova Pro              |
| チュートリアルコンポーネント | AgentCore Gatewayの作成とAgentCore Gatewayの呼び出し      |
| チュートリアル垂直領域 | クロス垂直領域                                             |
| 例の複雑さ        | 簡単                                                       |
| 使用SDK           | boto3                                                     |

チュートリアルの最初の部分では、いくつかのAmazonCore Gatewayターゲットを作成します

### チュートリアルアーキテクチャ
このチュートリアルでは、AWS Lambda関数で定義された操作をMCPツールに変換し、Bedrock AgentCore Gatewayでホストします。
デモンストレーションの目的で、Amazon Bedrockモデルを使用するStrands Agentを使用します
この例では、2つのツール（get_orderとupdate_order）を持つ非常にシンプルなエージェントを使用します。

## 前提条件

このチュートリアルを実行するには、以下が必要です：
* Jupyterノートブック（Pythonカーネル）
* uv
* AWS認証情報
* Amazon Cognito

## AgentCore Gatewayへの着信リクエストの認証設定
AgentCore Gatewayは、着信および発信認証による安全な接続を提供します。着信認証では、AgentCore Gatewayは呼び出し時に渡されるOAuthトークンを分析し、ゲートウェイ内のツールへのアクセスを許可するか拒否するかを決定します。ツールが外部リソースへのアクセスを必要とする場合、AgentCore GatewayはAPIキー、IAM、またはOAuthトークンによるアウトバウンド認証を使用して、外部リソースへのアクセスを許可または拒否できます。

インバウンド認証フローでは、エージェントまたはMCPクライアントがAgentCore Gateway内のMCPツールを呼び出し、OAuthアクセストークン（ユーザーのIdPから生成）を追加します。AgentCore GatewayはOAuthアクセストークンを検証し、インバウンド認証を実行します。

AgentCore Gateway上で動作するツールが外部リソースへのアクセスを必要とする場合、OAuthはゲートウェイ対象のリソース認証プロバイダーを使用して下流リソースの認証情報を取得します。AgentCore Gatewayは認証情報を呼び出し元に渡し、下流APIへのアクセスを可能にします。

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

In [1]:
# Amazon SageMakerノートブックを使用していない場合はAWS認証情報を設定
import os
# os.environ['AWS_ACCESS_KEY_ID'] = '' # アクセスキーを設定
# os.environ['AWS_SECRET_ACCESS_KEY'] = '' # シークレットキーを設定
os.environ['AWS_DEFAULT_REGION'] = os.environ.get('AWS_REGION', 'ap-northeast-1') # AWSリージョンを設定

In [2]:
import os
import sys

# 現在のスクリプトのディレクトリを取得
if '__file__' in globals():
    current_dir = os.path.dirname(os.path.abspath(__file__))
else:
    current_dir = os.getcwd()  # __file__が定義されていない場合のフォールバック（例：Jupyter）

# utils.pyを含むディレクトリに移動（1レベル上）
utils_dir = os.path.abspath(os.path.join(current_dir, '..'))

# sys.pathに追加
sys.path.insert(0, utils_dir)

# これでutilsをインポートできます
import utils

In [3]:
#### MCPツールに変換したいサンプルAWS Lambda関数を作成
lambda_resp = utils.create_gateway_lambda("lambda_function_code.zip")

if lambda_resp is not None:
    if lambda_resp['exit_code'] == 0:
        print("Lambda関数がARNで作成されました: ", lambda_resp['lambda_function_arn'])
    else:
        print("Lambda関数の作成が失敗しました。メッセージ: ", lambda_resp['lambda_function_arn'])

Reading code from zip file
Creating IAM role for lambda function
Attaching policy to the IAM role
Role 'gateway_lambda_iamrole' created successfully: arn:aws:iam::195049633937:role/gateway_lambda_iamrole
Creating lambda function
Lambda関数がARNで作成されました:  arn:aws:lambda:ap-northeast-1:195049633937:function:gateway_lambda


In [4]:
#### Gatewayが引き受けるIAMロールを作成
import utils
agentcore_gateway_iam_role = utils.create_agentcore_gateway_role("sample-lambdagateway")
print("Agentcore gatewayロールARN: ", agentcore_gateway_iam_role['Role']['Arn'])

attaching role policy agentcore-sample-lambdagateway-role
Agentcore gatewayロールARN:  arn:aws:iam::195049633937:role/agentcore-sample-lambdagateway-role


# Gatewayへのインバウンド認証用のAmazon Cognitoプールを作成

In [5]:
# Cognitoユーザープールを作成
import os
import boto3
import requests
import time
from botocore.exceptions import ClientError

REGION = os.environ['AWS_DEFAULT_REGION']
USER_POOL_NAME = "sample-agentcore-gateway-pool"
RESOURCE_SERVER_ID = "sample-agentcore-gateway-id"
RESOURCE_SERVER_NAME = "sample-agentcore-gateway-name"
CLIENT_NAME = "sample-agentcore-gateway-client"
SCOPES = [
    {"ScopeName": "gateway:read", "ScopeDescription": "読み取りアクセス"},
    {"ScopeName": "gateway:write", "ScopeDescription": "書き込みアクセス"}
]
scopeString = f"{RESOURCE_SERVER_ID}/gateway:read {RESOURCE_SERVER_ID}/gateway:write"

cognito = boto3.client("cognito-idp", region_name=REGION)

print("Cognitoリソースを作成または取得中...")
user_pool_id = utils.get_or_create_user_pool(cognito, USER_POOL_NAME)
print(f"ユーザープールID: {user_pool_id}")

utils.get_or_create_resource_server(cognito, user_pool_id, RESOURCE_SERVER_ID, RESOURCE_SERVER_NAME, SCOPES)
print("リソースサーバーを確保しました。")

client_id, client_secret  = utils.get_or_create_m2m_client(cognito, user_pool_id, CLIENT_NAME, RESOURCE_SERVER_ID)
print(f"クライアントID: {client_id}")

# ディスカバリーURLを取得
cognito_discovery_url = f'https://cognito-idp.{REGION}.amazonaws.com/{user_pool_id}/.well-known/openid-configuration'
print(cognito_discovery_url)

Cognitoリソースを作成または取得中...
Creating new user pool
Domain created as well
ユーザープールID: ap-northeast-1_fULHXdr1q
creating new resource server
リソースサーバーを確保しました。
creating new m2m client
クライアントID: 1vhcrj0k6b4obb8ngkip3h692n
https://cognito-idp.ap-northeast-1.amazonaws.com/ap-northeast-1_fULHXdr1q/.well-known/openid-configuration


# インバウンド認証用のAmazon Cognito AuthorizerでGatewayを作成

In [6]:
# CMKなしでCognito authorizerを使用してGatewayを作成。前のステップで作成したCognitoユーザープールを使用
gateway_client = boto3.client('bedrock-agentcore-control', region_name = os.environ['AWS_DEFAULT_REGION'])
auth_config = {
    "customJWTAuthorizer": { 
        "allowedClients": [client_id],  # クライアントはCognitoで設定されたClientIdと一致する必要があります。例: 7rfbikfsm51j2fpaggacgng84g
        "discoveryUrl": cognito_discovery_url
    }
}
create_response = gateway_client.create_gateway(name='TestGWforLambda',
    roleArn = agentcore_gateway_iam_role['Role']['Arn'], # IAMロールはGatewayの作成/リスト/取得/削除の権限を持っている必要があります
    protocolType='MCP',
    authorizerType='CUSTOM_JWT',
    authorizerConfiguration=auth_config, 
    description='AWS Lambdaターゲットタイプを使用したAgentCore Gateway'
)
print(create_response)
# GatewayTarget作成に使用されるGatewayIDを取得
gatewayID = create_response["gatewayId"]
gatewayURL = create_response["gatewayUrl"]
print(gatewayID)

{'ResponseMetadata': {'RequestId': '926d9f06-3bc9-4158-b225-2da6b20ad367', 'HTTPStatusCode': 202, 'HTTPHeaders': {'date': 'Sun, 04 Jan 2026 08:13:18 GMT', 'content-type': 'application/json', 'content-length': '856', 'connection': 'keep-alive', 'x-amzn-requestid': '926d9f06-3bc9-4158-b225-2da6b20ad367', 'x-amzn-remapped-x-amzn-requestid': 'd1467867-f6bf-4917-92df-367ef55b7e39', 'x-amzn-remapped-content-length': '856', 'x-amzn-remapped-connection': 'keep-alive', 'x-amz-apigw-id': 'WpocvGgvNjMEFZg=', 'x-amzn-trace-id': 'Root=1-695a211d-2686736e0b6826283daad6a8', 'x-amzn-remapped-date': 'Sun, 04 Jan 2026 08:13:18 GMT'}, 'RetryAttempts': 0}, 'gatewayArn': 'arn:aws:bedrock-agentcore:ap-northeast-1:195049633937:gateway/testgwforlambda-ewgfmtogw9', 'gatewayId': 'testgwforlambda-ewgfmtogw9', 'gatewayUrl': 'https://testgwforlambda-ewgfmtogw9.gateway.bedrock-agentcore.ap-northeast-1.amazonaws.com/mcp', 'createdAt': datetime.datetime(2026, 1, 4, 8, 13, 18, 63503, tzinfo=tzutc()), 'updatedAt': date

# AWS Lambdaターゲットを作成し、MCPツールに変換

In [7]:
# 以下のAWS Lambda関数ARNを置き換えてください
lambda_target_config = {
    "mcp": {
        "lambda": {
            "lambdaArn": lambda_resp['lambda_function_arn'], # これをあなたのAWS Lambda関数ARNに置き換えてください
            "toolSchema": {
                "inlinePayload": [
                    {
                        "name": "get_order_tool",
                        "description": "注文を取得するツール",
                        "inputSchema": {
                            "type": "object",
                            "properties": {
                                "orderId": {
                                    "type": "string"
                                }
                            },
                            "required": ["orderId"]
                        }
                    },                    
                    {
                        "name": "update_order_tool",
                        "description": "orderIdを更新するツール",
                        "inputSchema": {
                            "type": "object",
                            "properties": {
                                "orderId": {
                                    "type": "string"
                                }
                            },
                            "required": ["orderId"]
                        }
                    }
                ]
            }
        }
    }
}

credential_config = [ 
    {
        "credentialProviderType" : "GATEWAY_IAM_ROLE"
    }
]
targetname='LambdaUsingSDK'
response = gateway_client.create_gateway_target(
    gatewayIdentifier=gatewayID,
    name=targetname,
    description='SDKを使用したLambdaターゲット',
    targetConfiguration=lambda_target_config,
    credentialProviderConfigurations=credential_config)

# Strands AgentからBedrock AgentCore Gatewayを呼び出す

Strandsエージェントは、Model Context Protocol（MCP）仕様を実装するBedrock AgentCore Gatewayを通じてAWSツールとシームレスに統合します。この統合により、AIエージェントとAWSサービス間の安全で標準化された通信が可能になります。

Bedrock AgentCore Gatewayは、基本的なMCP API（ListToolsとInvokeTools）を公開するプロトコル準拠のGatewayとして機能します。これらのAPIにより、MCP準拠のクライアントやSDKは、安全で標準化された方法で利用可能なツールを発見し、対話できます。StrandsエージェントがAWSサービスにアクセスする必要がある場合、これらのMCP標準化エンドポイントを使用してGatewayと通信します。

Gatewayの実装は、[MCP Authorization仕様](https://modelcontextprotocol.org/specification/draft/basic/authorization)に厳密に準拠しており、堅牢なセキュリティとアクセス制御を確保しています。これは、Strandsエージェントによるすべてのツール呼び出しが認証ステップを通過することを意味し、強力な機能を有効にしながらセキュリティを維持します。

例えば、StrandsエージェントがMCPツールにアクセスする必要がある場合、まずListToolsを呼び出して利用可能なツールを発見し、次にInvokeToolsを使用して特定のアクションを実行します。Gatewayは、必要なすべてのセキュリティ検証、プロトコル変換、サービス対話を処理し、プロセス全体をシームレスで安全にします。

このアーキテクチャアプローチは、MCP仕様を実装する任意のクライアントまたはSDKがGatewayを通じてAWSサービスと対話できることを意味し、AIエージェント統合のための汎用的で将来性のあるソリューションとなります。

![Gatewayを呼び出すStrandsエージェント](images/strands-lambda-gateway.png)

# インバウンド認証用のAmazon Cognitoからアクセストークンをリクエスト

In [8]:
import time
time.sleep(10)

In [9]:
print("Amazon Cognito authorizerからアクセストークンをリクエスト中...ドメイン名の伝播が完了するまでしばらく失敗する可能性があります")
token_response = utils.get_token(user_pool_id, client_id, client_secret,scopeString,REGION)
token = token_response["access_token"]
print("トークンレスポンス:", token)

Amazon Cognito authorizerからアクセストークンをリクエスト中...ドメイン名の伝播が完了するまでしばらく失敗する可能性があります
1vhcrj0k6b4obb8ngkip3h692n
トークンレスポンス: eyJraWQiOiJaSXlzRmExb09KWUxvYXRVcTN3TTFPTXg4Wmo5ZFRyMWxDRW56cUhtRFBRPSIsImFsZyI6IlJTMjU2In0.eyJzdWIiOiIxdmhjcmowazZiNG9iYjhuZ2tpcDNoNjkybiIsInRva2VuX3VzZSI6ImFjY2VzcyIsInNjb3BlIjoic2FtcGxlLWFnZW50Y29yZS1nYXRld2F5LWlkXC9nYXRld2F5OndyaXRlIHNhbXBsZS1hZ2VudGNvcmUtZ2F0ZXdheS1pZFwvZ2F0ZXdheTpyZWFkIiwiYXV0aF90aW1lIjoxNzY3NTE0NDY4LCJpc3MiOiJodHRwczpcL1wvY29nbml0by1pZHAuYXAtbm9ydGhlYXN0LTEuYW1hem9uYXdzLmNvbVwvYXAtbm9ydGhlYXN0LTFfZlVMSFhkcjFxIiwiZXhwIjoxNzY3NTE4MDY4LCJpYXQiOjE3Njc1MTQ0NjgsInZlcnNpb24iOjIsImp0aSI6ImM3Yzg5ZGExLWEyYzYtNDY2NS05MGRiLTlkNGFlYzZkZTc2MiIsImNsaWVudF9pZCI6IjF2aGNyajBrNmI0b2JiOG5na2lwM2g2OTJuIn0.s5HSoHCwuY7t4CamGiUKDxZ66eVjSIXB7hT3xGqJECVcWoRrsSDk8AeHj_sp28EAC5jrGk-l380u-ucdBYXHSV8kviEz5FuvlUE4qC56Opl_j7rSFK3U4hOPMqwUPVCchxIwh-8Tt2mzx7l2ywiWlpPdbakrnzAh7p48Jv2Aq9C6ppXYUdKe8AIQQeHj4kdRM239f3KNOL1MLM9QA7HFYRA_6GmLlLQINUw1vRUpQADPCS9HSML1DPZZrTSByOOM43Rhj_Vt4MK7U

# Bedrock AgentCore Gatewayを使用してAWS LambdaのMCPツールを呼び出すStrandsエージェント

In [12]:
from strands.models import BedrockModel
from mcp.client.streamable_http import streamablehttp_client 
from strands.tools.mcp.mcp_client import MCPClient
from strands import Agent

def create_streamable_http_transport():
    return streamablehttp_client(gatewayURL,headers={"Authorization": f"Bearer {token}"})

client = MCPClient(create_streamable_http_transport)

## ~/.aws/credentialsに設定されたIAM認証情報はBedrockモデルへのアクセス権限を持っている必要があります
yourmodel = BedrockModel(
    model_id="apac.amazon.nova-pro-v1:0",
    temperature=0.7,
)

In [13]:
from strands import Agent
import logging


# ルートstrandsロガーを設定。問題をデバッグしている場合はDEBUGに変更してください。
logging.getLogger("strands").setLevel(logging.INFO)

# ログを表示するためのハンドラーを追加
logging.basicConfig(
    format="%(levelname)s | %(name)s | %(message)s", 
    handlers=[logging.StreamHandler()]
)

with client:
    # listToolsを呼び出す
    tools = client.list_tools_sync()
    # モデルとツールでエージェントを作成
    agent = Agent(model=yourmodel,tools=tools) ## お好みのモデルに置き換えることができます
    print(f"エージェントに読み込まれたツールは {agent.tool_names}")
    # print(f"エージェント内のツール設定は {agent.tool_config}")
    # サンプルプロンプトでエージェントを呼び出す。これはMCP listToolsを呼び出し、LLMがアクセスできるツールのリストを取得するだけです。以下は実際にはツールを呼び出しません。
    agent("こんにちは、利用可能なすべてのツールをリストできますか")
    # サンプルプロンプトでエージェントを呼び出し、ツールを呼び出してレスポンスを表示
    agent("注文ID 123の注文ステータスを確認し、ツールからの正確なレスポンスを表示してください")
    # MCPツールを明示的に呼び出す。MCPツール名と引数は、AWS Lambda関数またはOpenAPI/Smithy APIと一致する必要があります
    result = client.call_tool_sync(
    tool_use_id="get-order-id-123-call-1", # これを一意の識別子に置き換えることができます。
    name=targetname+"___get_order_tool", # これはAWS Lambdaターゲットタイプに基づくツール名です。ターゲット名に基づいて変更されます
    arguments={"orderId": "123"}
    )
    # MCPツールのレスポンスを表示
    print(f"ツール呼び出し結果: {result['content'][0]['text']}")


エージェントに読み込まれたツールは ['LambdaUsingSDK___get_order_tool', 'LambdaUsingSDK___update_order_tool']
<thinking> The user has requested a list of all available tools. I should provide a list of the tools that are available for use. </thinking>

こちらが利用可能なツールのリストです:
1. LambdaUsingSDK___get_order_tool - 注文を取得するツール
2. LambdaUsingSDK___update_order_tool - orderIdを更新するツール<thinking> The user wants to check the order status for order ID 123. I should use the `LambdaUsingSDK___get_order_tool` to retrieve the order status. </thinking>


Tool #1: LambdaUsingSDK___get_order_tool
<thinking> The tool has returned the order status for order ID 123. I should extract the relevant information from the tool result and present it to the user. </thinking>

ツールからの正確なレスポンスは次のとおりです:
```
Order Id 123 is in shipped status
```ツール呼び出し結果: {"statusCode":200,"body":"Order Id 123 is in shipped status"}


**問題: 以下のセルを実行中に以下のエラーが発生した場合、pydanticとpydantic-coreのバージョン間の非互換性を示しています。**

```
TypeError: model_schema() got an unexpected keyword argument 'generic_origin'
```
**解決方法**

pydantic==2.7.2とpydantic-core 2.27.2の両方が互換性があることを確認する必要があります。完了したらカーネルを再起動してください。

# クリーンアップ

IAMロール、IAMポリシー、認証情報プロバイダー、AWS Lambda関数、Cognitoユーザープール、S3バケットなどの追加リソースも作成されており、クリーンアップの一部として手動で削除する必要がある場合があります。これは実行する例によって異なります。

## Gatewayを削除（オプション）

In [14]:
import utils
utils.delete_gateway(gateway_client,gatewayID)

Deleting all targets for gateway testgwforlambda-ewgfmtogw9
Deleting target  V2CGEDI5SD
Deleting gateway  testgwforlambda-ewgfmtogw9
